# Reproduce the N1 waste-classification pipeline on Google Colab

Repository: https://github.com/kang-prayit/n1-trustworthy-waste-classification · Archived release (Zenodo): https://doi.org/10.5281/zenodo.23696855

This notebook was written for the public release. It is **not** the notebook that produced the paper's
results: that one contained account-specific paths. It runs the same commands as the README. Use a GPU
runtime (*Runtime → Change runtime type → GPU*). The paper's evaluation runs used an A100.

Options:
- **A. Check the published results.** Run the unit tests and regenerate the paper figures from the tables in the repository; no GPU needed.
- **B. Full pipeline.** Download the datasets, install the published splits, then train, evaluate and analyse. 252 runs take about 31 GPU-hours on a mix of RTX 4070 and A100 GPUs. Use `--blocks` / `--max-runs` to do a part of it.

In [ ]:
!git clone https://github.com/kang-prayit/n1-trustworthy-waste-classification.git
%cd n1-trustworthy-waste-classification
!nvidia-smi -L || true

Colab already ships a recent PyTorch build. The paper used PyTorch 2.11.0 + CUDA 13.0 and timm 1.0.30. The
next cell installs the remaining requirements and keeps Colab's torch. To match the paper exactly, install
`torch==2.11.0` from the cu130 wheel index instead (see `requirements.txt`).

In [ ]:
!pip install -q timm==1.0.30 imagehash==4.3.2 pyyaml tqdm scikit-learn scipy pandas matplotlib pillow pytest
!python -m pytest -q tests

Optional: keep `data/`, `work/` and `outputs/` on Google Drive so a disconnected session can resume.
Mount Drive, then set `ROOT` to a folder of your choice. The paths go into a local config; no code changes are needed.

In [ ]:
ROOT = "/content/n1_data"   # e.g. a folder on your mounted Google Drive
# from google.colab import drive; drive.mount('/content/drive')
cfg = f"""base: full.yaml
data_root: {ROOT}/data
work_dir: {ROOT}/work
out_dir: {ROOT}/outputs
log_dir: {ROOT}/logs
"""
open("configs/local.yaml", "w").write(cfg)
print(cfg)

## A. Regenerate the paper figures from the published results
Figs 4, 6, S3 and S4 also need the per-run probabilities (`probs.tar`), available from the corresponding author on reasonable request.

In [ ]:
!python scripts/make_paper_figures.py --out build/paper_figures                       # needs no probabilities
# !python scripts/make_paper_figures.py --probs-tar probs.tar --out build/paper_figures  # all figures

## B. Full pipeline
Download the datasets (please respect their licences, see README), quarantine the 3 corrupt TrashBox files, and install the published splits.

In [ ]:
!python download_data.py --data-root {ROOT}/data
!python scripts/quarantine_corrupt.py --data-root {ROOT}/data
!python scripts/install_splits.py --config configs/local.yaml

In [ ]:
!python run_all.py --config configs/local.yaml --estimate
# !python run_all.py --config configs/local.yaml --stages prepare train      # resumable; re-run after a disconnect
# !python n1_finalize.py --config configs/local.yaml                          # eval, cross, gradcam, analyze, pack